# Security Tooling for Python Projects

A leaked API key in git history is practically permanent once pushed. GitHub indexes commit history, and forks preserve deleted content — rotating the key is the only reliable fix. The two distinct threat classes are (1) *secrets in code*, where credentials end up in committed files or commit history, and (2) *vulnerable dependencies*, where a package already in the lockfile is later found to have a known CVE. These require different defenses.

This notebook documents a defense-in-depth stack addressing both: (1) a pre-commit hook catches secrets before they leave the machine, (2) dependency vulnerability scanning triggers when dependencies change, (3) CI gates block secrets and vulnerable packages on every pull request, and (4) runtime secret management prevents secrets from entering source files in the first place. Additional topics covered include workflow hardening via SHA-pinned Actions and `zizmor` auditing, and static analysis with Bandit for Python-specific security issues. The appendix covers recovery from a leak.

## Pre-commit Secrets Scanner

The primary tool is **Gitleaks** — a fast, Go-based scanner with built-in rules for over 150 secret types (OpenAI keys, AWS credentials, GitHub tokens, etc.). For teams needing a Python-installable alternative or large false-positive baseline management, **detect-secrets** is an option, but Gitleaks is preferred here for its zero-Python-dependency installation and broad default ruleset.

**Install.** Add `pre-commit` as a dev dependency and register the hook:

```{.bash filename="$ (local)"}
uv add --dev pre-commit
pre-commit install
```

`pre-commit` manages hook environments in isolation — it downloads and installs Gitleaks automatically via the hook config, so no separate binary installation is needed. The hook runs on every `git commit`.

**Configure.** The `.pre-commit-config.yaml` file at the repo root declares which hooks to run:

```{.yaml filename=".pre-commit-config.yaml"}
repos:
  - repo: https://github.com/gitleaks/gitleaks
    rev: v8.24.2
    hooks:
      - id: gitleaks
```

Pin `rev` to a specific release tag rather than `main` — mutable branch refs mean the hook silently changes behavior on the next `pre-commit autoupdate`.

**Run (audit existing repo).** `pre-commit run gitleaks --all-files` scans all tracked files, not just staged ones. This is useful for an initial audit of an existing repository:

```{.bash filename="$ (local)"}
pre-commit run gitleaks --all-files
```

A finding looks like:

```
Finding:     secret_key="sk-proj-REDACTED..."
Secret:      sk-proj-REDACTED...
RuleID:      openai-api-key
Entropy:     4.12
File:        notebooks/agents/01.ipynb
Line:        42
Fingerprint: notebooks/agents/01.ipynb:openai-api-key:42
```

The `Fingerprint` field is a stable identifier for the finding — it is used to permanently suppress false positives.

**False positives.** Three mechanisms handle legitimate strings that resemble secrets: (1) inline suppression via `# gitleaks:allow` on the offending line, (2) a `.gitleaksignore` file containing fingerprints to suppress permanently, and (3) a custom `.gitleaks.toml` for project-wide allow-listing of paths or patterns.

A minimal `.gitleaks.toml` extending the default ruleset:

```{.toml filename=".gitleaks.toml"}
[extend]
useDefault = true

[allowlist]
paths = [
    "tests/fixtures/",
]
```

The `useDefault = true` line ensures all built-in rules remain active — the config only adds to them.

:::{.callout-caution}
Do not habitually skip the hook with `SKIP=gitleaks git commit`. Use it once for a confirmed false positive, then immediately add the fingerprint to `.gitleaksignore` so the suppression is tracked in version control.
:::

## Dependency Vulnerability Checking

**pip-audit** queries the Python Packaging Advisory Database (PyPA) and the GitHub Advisory Database for CVEs in installed packages. **Dependabot** complements it by automatically opening pull requests to upgrade vulnerable packages. The two tools are used together: Dependabot fixes the problem proactively, pip-audit in CI enforces the fix is deployed before code ships.

**Timing trade-offs.** Where to run pip-audit involves a friction/freshness trade-off:

| Option | Dev friction | Freshness | Recommendation |
|---|---|---|---|
| Pre-commit hook | High — blocks every commit | At commit time | Avoid — network-bound, slows all commits |
| Manual after `uv add` | None | At dep change | Good habit; run `pip-audit --locked .` |
| Scheduled CI workflow | None | Weekly | Best for background scanning |
| Dependabot | None | Near-real-time | Best for automated upgrade PRs |

The recommended approach is to add Dependabot for automated security PRs, add a scheduled CI workflow for periodic audits, and run `pip-audit --locked .` manually after `uv add`. Do NOT add pip-audit as a pre-commit hook.

**Install and run locally.**

```{.bash filename="$ (local)"}
uv tool install pip-audit
```

Useful invocations:

```{.bash filename="$ (local)"}
pip-audit --locked .                                      # audit the locked dependency tree
pip-audit --locked . --desc --aliases                    # include CVE descriptions and aliases
pip-audit --locked . --ignore-vuln GHSA-w596-4wvx-j9j6  # suppress a known false positive
```

Advisory IDs use three prefixes: `PYSEC-` (PyPI Advisory Database), `CVE-` (NIST National Vulnerability Database), and `GHSA-` (GitHub Advisory Database). A finding looks like:

```
Found 1 known vulnerability in 1 package
Name    Version ID                  Fix Versions
------- ------- ------------------- ------------
pillow  9.0.0   GHSA-56pw-mpj4-fxjw 9.0.1
```

:::{.callout-note}
`--locked` audits `uv.lock`, not the activated environment — it catches vulnerabilities before `uv sync` installs them. Omitting `--locked` audits the live environment instead, which may differ from the lockfile.
:::

**Dependabot.** Enable automated dependency upgrade PRs by adding `.github/dependabot.yml`:

```{.yaml filename=".github/dependabot.yml"}
version: 2
updates:
  - package-ecosystem: "uv"
    directory: "/"
    schedule:
      interval: "weekly"
    groups:
      patch-and-minor:
        update-types: ["patch", "minor"]

  - package-ecosystem: "github-actions"
    directory: "/"
    schedule:
      interval: "weekly"
```

Notice that `package-ecosystem: "uv"` is the correct value for `uv`-managed projects — not `"pip"`. The `groups` block batches patch and minor upgrades into a single PR rather than one PR per package, reducing noise.

## CI Security Checks

Two CI workflows enforce security gates on every push and pull request: a secrets scan using `gitleaks/gitleaks-action@v2` and a dependency audit using `pypa/gh-action-pip-audit@v1`.

**Secrets scan on every push and PR.** The following workflow runs Gitleaks against the full commit history on every push to any branch and every pull request:

```{.yaml filename=".github/workflows/security-secrets.yml"}
name: Secrets Scan

on:
  push:
    branches: ["**"]
  pull_request:
    branches: ["**"]

jobs:
  gitleaks:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0           # <1>
      - uses: gitleaks/gitleaks-action@v2
        env:
          GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}
```


1. Gitleaks scans git history — full history requires `fetch-depth: 0`. The default shallow clone misses all commits beyond the latest, allowing a secret introduced in an older commit to pass undetected.

For personal repositories, no `GITLEAKS_LICENSE` secret is needed. GitHub organization repositories require a license key set as a repository secret.

**Scheduled dependency audit.** The following workflow runs pip-audit on a weekly schedule and also triggers immediately when the lockfile or manifest changes:

```{.yaml filename=".github/workflows/security-audit.yml"}
name: Dependency Audit

on:
  schedule:
    - cron: "0 9 * * 1"           # <1>
  push:
    paths:
      - "uv.lock"
      - "pyproject.toml"          # <2>

jobs:
  pip-audit:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v7
      - uses: pypa/gh-action-pip-audit@v1.1.0
        with:
          inputs: "."
          args: "--locked"
```


1. Runs every Monday at 09:00 UTC regardless of commits — catches new CVEs disclosed since the last code change.
2. Also triggers immediately on any push that modifies the dependency lockfile or manifest, so a `uv add` that introduces a known vulnerability fails in CI before merging.

:::{.callout-note}
Dependabot creates PRs to upgrade vulnerable packages — it is asynchronous and proactive. pip-audit in CI blocks a merge if a vulnerability exists at that moment. Both are necessary: Dependabot surfaces the fix, pip-audit enforces that the fix is deployed before code ships.
:::

## GitHub Actions Workflow Hardening

The supply-chain risk in GitHub Actions is subtle: `uses: actions/checkout@v4` pins to a tag, but tags are mutable pointers — a compromised maintainer account can update `v4` to point to a different, malicious commit. Pinning to a full commit SHA eliminates this vector entirely:

```yaml
- uses: actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683  # v4.2.2
```

The inline comment preserves human readability while the SHA provides immutability. Tools like `zizmor` and Dependabot's `github-actions` ecosystem can automate the maintenance of SHA pins.

:::{.callout-important}
Third-party Actions execute arbitrary code in the CI runner with access to `secrets.GITHUB_TOKEN` and any secrets loaded into the job. A malicious action can exfiltrate credentials or push code to the repository. Pin to commit SHAs for any action in a security-sensitive workflow.
:::

**Install and run zizmor.**

```{.bash filename="$ (local)"}
uv tool install zizmor
zizmor .github/workflows/
```

`zizmor` audits workflow YAML files for: unpinned action refs, over-permissioned `permissions:` blocks, `pull_request_target` misuse (a common privilege escalation vector), and expression injection vulnerabilities where untrusted input like `github.event.pull_request.title` flows into a `run:` step.

Example output:

```
.github/workflows/publish.yml:
  1 finding(s):

  ⚠ unpinned-uses [medium]
    Line 17: uses: quarto-dev/quarto-actions/setup@v2
    Fix: pin to commit SHA
```

**CODEOWNERS.** Requiring a code-owner review for any workflow file change prevents an unauthorized contributor from adding a malicious step to an existing workflow:

```{.bash filename=".github/CODEOWNERS"}
.github/workflows/  @your-github-username
```

With this in place, a pull request touching any file under `.github/workflows/` cannot be merged without an approving review from the listed owner, even if the repository allows auto-merge for other PRs.

**Permissions: principle of least privilege.** By default, GitHub Actions grants the `GITHUB_TOKEN` `read` access to all scopes. Scope permissions at the job level so that only the job that needs write access gets it:

```yaml
jobs:
  build-deploy:
    permissions:
      contents: write      # only this job needs write; others default to read
```

Setting `permissions` at the workflow level with a permissive default and overriding per-job is less safe than starting from a restrictive default. A top-level `permissions: read-all` with explicit job-level grants is the recommended structure.

## Static Code Security Analysis

**Bandit** analyzes Python source for common security issues: hardcoded passwords (`B105`–`B107`), shell injection via `subprocess` with `shell=True` (`B602`/`B603`), use of `assert` for authentication checks (`B101`), and unsafe deserialization via `pickle` (`B301`). It is a static analysis tool — it does not execute code.

**Install and run.**

```{.bash filename="$ (local)"}
uv add --dev bandit
bandit -r src/ -ll        # medium and high severity only
```

The `-ll` flag filters to medium and high severity, reducing noise from low-severity informational findings during initial setup.

**Configure in `pyproject.toml`.**

```{.toml filename="pyproject.toml"}
[tool.bandit]
exclude_dirs = ["tests", "notebooks"]
skips = ["B101"]          # assert statements are acceptable in tests
```

Notebooks are excluded because they routinely use patterns that Bandit flags — hardcoded example values, `subprocess` calls for shell demos, and `pickle` for model checkpointing. Bandit is most useful scoped to `src/`.

**Pre-commit hook.**

```{.yaml filename=".pre-commit-config.yaml"}
- repo: https://github.com/PyCQA/bandit
  rev: 1.8.3
  hooks:
    - id: bandit
      args: ["-c", "pyproject.toml"]
```

The `-c pyproject.toml` argument ensures the hook respects the `[tool.bandit]` configuration above, including the `exclude_dirs` list.

**The pickle risk.** `pickle.load()` executes arbitrary Python code embedded in the serialized file. Loading a model checkpoint from an untrusted source is equivalent to running the file author's code with local filesystem permissions. The `weights_only=True` parameter in `torch.load` restricts deserialization to tensor data only, blocking code execution:

In [ ]:
import torch

# Unsafe: arbitrary code execution if the file is malicious
# model_state = torch.load("checkpoint.pt")

# Safe: weights_only=True restricts deserialization to tensors only
model_state = torch.load("checkpoint.pt", weights_only=True)

:::{.callout-caution}
`pickle.load(f)` will execute any Python code embedded in the file. Loading a Hugging Face checkpoint or a downloaded notebook model without auditing is equivalent to running the model author's code with local permissions. Use `weights_only=True` with `torch.load`, or switch to `safetensors` for model weights.
:::

## Runtime Secret Management

API keys for OpenAI, Groq, AWS, and similar services must never appear in notebook source, `pyproject.toml`, or any committed file. The standard pattern is a `.env` file on disk, gitignored, loaded at runtime via `python-dotenv`.

**Install.**

```{.bash filename="$ (local)"}
uv add python-dotenv
```

**`.env` file format** (never committed — add `.env` to `.gitignore`):

```{.bash filename=".env"}
OPENAI_API_KEY=sk-proj-...
GROQ_API_KEY=gsk_...
AWS_ACCESS_KEY_ID=AKIA...
AWS_SECRET_ACCESS_KEY=...
```

Commit a `.env.example` file alongside it with the same keys but empty values — this documents what variables are expected without exposing real credentials:

```{.bash filename=".env.example"}
OPENAI_API_KEY=
GROQ_API_KEY=
AWS_ACCESS_KEY_ID=
AWS_SECRET_ACCESS_KEY=
```

Verify that `.env` is gitignored with:

```{.bash filename="$ (local)"}
git check-ignore -v .env
```

If the command returns the path (e.g. `.gitignore:5:.env`), git will not track it. No output means it is not ignored.

**Load at runtime.** Call `load_dotenv()` at the top of any notebook or script that needs credentials:

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env from the working directory

api_key = os.environ["OPENAI_API_KEY"]  # raises KeyError if missing

`os.environ["KEY"]` raises `KeyError` if the variable is missing — it fails loudly at startup rather than silently failing later with an unhelpful error. `os.getenv("KEY", "default")` silently falls back to a hardcoded value, which is the wrong behavior for secrets: a misconfigured environment would proceed with a wrong or empty key.

**GitHub Actions secrets.** In CI, `.env` does not exist — secrets are injected via the GitHub Actions secret store and surfaced as environment variables in the job:

```yaml
- name: Run notebook
  env:
    OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
  run: jupyter nbconvert --to notebook --execute notebook.ipynb
```

GitHub Actions automatically masks secret values in log output — any log line containing the raw secret value is replaced with `***`.

:::{.callout-note}
`.env` files are plaintext on disk — anyone with filesystem access can read them. They are appropriate for local development only. For shared environments, use the platform's secret store: GitHub Actions secrets, AWS Secrets Manager, 1Password CLI, or similar.
:::

## Appendix: Recovering from a Leaked Secret

If a secret is committed and pushed, the steps are (1) rotate the credential immediately — before anything else — and (2) assume it was seen. GitHub indexes commit history; forks preserve deleted content; cached copies may exist in search engines or scanning tools. History rewriting does not undo exposure retroactively.

**Rotation is the priority.** Removing the secret from history without rotating first leaves the window of exposure open while history rewriting is in progress. Rotate the key, then clean up history.

**Purging from git history.** `git filter-repo` is the modern replacement for `BFG Repo Cleaner` and `git filter-branch`. It rewrites all reachable commits:

```{.bash filename="$ (local)"}
# Install
pip install git-filter-repo

# Remove a specific file from all history
git filter-repo --path secrets.txt --invert-paths

# Replace a specific secret value with a placeholder in all history
git filter-repo --replace-text <(echo "sk-proj-ACTUAL_KEY==>REDACTED")

# Force-push all refs after rewrite
git push --force --all
git push --force --tags
```

After a force-push, all collaborators must re-clone — their local history is now incompatible with the rewritten remote. Communicate this before proceeding.

**GitHub secret scanning.** GitHub's Security tab (available free on public repositories) shows secret scanning alerts. Check it after a suspected leak to see what GitHub itself detected — this confirms whether the exposure was indexed.

:::{.callout-important}
Force-pushing does not immediately remove the commit from GitHub's servers. GitHub caches commits for a period after deletion. For sensitive credentials, contact GitHub Support after the force-push to request a cache purge.
:::

---

■